# Predictive Invoice Analytics
**Mahmoud Abusahyoun | Data Analytics coursework, HTU**

This rerun uses all 532,057 supplied rows. It retains the original coursework's smoothing, linear regression, decision tree, SimpleRNN and LSTM approaches. The exact original notebooks are preserved separately in the backup.

Linear/tree experiments use a seeded 60%/20%/20% training/validation/test split. Categorical encoding and feature selection fit training data only. Methods are compared on validation data; the selected method is refitted on development data and evaluated once on the test set. Numeric errors are in the workbook's unspecified monetary units.

`days_overdue` may only be known after invoice issuance. These are retrospective regressions unless feature availability is established. Moving averages are smoothing curves, not held-out forecasts. Neural sequence experiments model consecutive chronological invoice rows across different invoices; they are exploratory and do not establish forecasting performance for a recurring customer or entity. They use a separate chronological split and cannot be ranked directly against the random-split models. Historical coursework also inspected test results, so this is retrospective validation, not an independent external study.

In [1]:
from pathlib import Path
import json, os, pickle, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_selection import SequentialFeatureSelector, SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
df = pd.read_csv('data/invoices_model_data.csv.gz', float_precision='round_trip')
assert df.shape == (532057, 8)
assert not df.isna().any().any()
print('Rows:', len(df), 'Columns:', list(df.columns))
df.head()

Rows: 532057 Columns: ['issue_date', 'collection_type', 'invoice_sub_type', 'invoice_category', 'invoice_discount_amount', 'invoice_tax_amount', 'days_overdue', 'expected_payment_amount']


,issue_date,collection_type,invoice_sub_type,invoice_category,invoice_discount_amount,invoice_tax_amount,days_overdue,expected_payment_amount
0,1708014393,CREDIT,EXPORT,GENERAL_SALES,0.0,106.23,0,649.84
1,1749262653,CASH,LOCAL,INCOME,0.0,135.67,12,1437.72
2,1642622798,CASH,LOCAL,INCOME,0.0,96.66,12,1216.49
3,1706758339,CREDIT,EXPORT,SPECIAL_SALES,0.0,81.02,0,1817.24
4,1694894569,CREDIT,AREA_OF_DEVELOPMENT,INCOME,0.0,120.53,0,1244.63


## 1. Statistical smoothing

In [2]:
dates = pd.to_datetime(df.issue_date, unit='s', utc=True)
daily = pd.Series(df.expected_payment_amount.to_numpy(), index=dates).resample('D').mean().dropna()
# Daily means keep the visual readable and avoid plotting half a million overlapping points.
fig, axes = plt.subplots(2,1, figsize=(12,8))
daily.plot(ax=axes[0], alpha=.5, label='Daily mean')
for window in [10,55,95]: daily.rolling(window).mean().plot(ax=axes[0], label=f'SMA {window} days')
axes[0].legend(); axes[0].set_title('Daily expected payments: simple moving averages')
daily.plot(ax=axes[1], alpha=.4, label='Daily mean')
for alpha in [.3,.6,.9]: daily.ewm(alpha=alpha).mean().plot(ax=axes[1], label=f'EMA alpha={alpha}')
axes[1].legend(); axes[1].set_title('Daily expected payments: exponential moving averages')
plt.tight_layout(); plt.show()
daily.to_csv(RESULTS / 'daily_mean_payments.csv', header=['expected_payment_amount'])

<Figure size 1200x800 with 2 Axes>

## 2. Training, validation and test partitions

In [3]:
features = ['collection_type','invoice_sub_type','invoice_category','invoice_discount_amount','invoice_tax_amount','days_overdue']
categorical = features[:3]
def encode_frame(frame, encoder, fit=False):
    out = frame[features].copy()
    values = encoder.fit_transform(out[categorical]) if fit else encoder.transform(out[categorical])
    out[categorical] = values
    return out.astype(float)
dev_idx, test_idx = train_test_split(np.arange(len(df)), test_size=.2, random_state=SEED)
train_idx, val_idx = train_test_split(dev_idx, test_size=.25, random_state=SEED)
assert not set(train_idx) & set(val_idx) and not set(dev_idx) & set(test_idx)
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train = encode_frame(df.iloc[train_idx], encoder, fit=True)
X_val = encode_frame(df.iloc[val_idx], encoder)
y_train = df.expected_payment_amount.iloc[train_idx].to_numpy()
y_val = df.expected_payment_amount.iloc[val_idx].to_numpy()
print('Train / validation / test:', len(train_idx), len(val_idx), len(test_idx))
def metrics(y, pred):
    return {'MAE':float(mean_absolute_error(y,pred)), 'MSE':float(mean_squared_error(y,pred)), 'RMSE':float(np.sqrt(mean_squared_error(y,pred))), 'R2':float(r2_score(y,pred))}
cv = KFold(3, shuffle=True, random_state=SEED)
rows=[]; candidates={}

Train / validation / test: 319233 106412 106412


## 3. Feature selection and regression
Forward selection uses three-fold cross-validation within the fitting partition. The tree is regularized to depth 12 with at least 20 rows per leaf. Ordinal category codes are a simplification inherited from the coursework; they can impose artificial category order, especially for linear regression. Test labels are not used for selecting features or choosing a model.

In [4]:
estimators = {'Linear':LinearRegression(), 'Tree':DecisionTreeRegressor(max_depth=12,min_samples_leaf=20,random_state=SEED)}
for model_name, estimator in estimators.items():
    for selection in ['Forward','KBest']:
        selector = (SequentialFeatureSelector(clone(estimator),n_features_to_select=3,cv=cv,n_jobs=1)
                    if selection == 'Forward' else SelectKBest(f_regression,k=3))
        selector.fit(X_train,y_train)
        model = clone(estimator).fit(selector.transform(X_train),y_train)
        pred = model.predict(selector.transform(X_val))
        name = model_name + '_' + selection
        row={'model':name, 'selected_features':','.join(X_train.columns[selector.get_support()]), **metrics(y_val,pred)}
        rows.append(row); candidates[name]=(selector,model)
        print(name, row, flush=True)
comparison = pd.DataFrame(rows).sort_values('RMSE')
comparison.to_csv(RESULTS / 'validation_comparison.csv',index=False)
comparison

Linear_Forward {'model': 'Linear_Forward', 'selected_features': 'collection_type,invoice_tax_amount,days_overdue', 'MAE': 488.74615440771123, 'MSE': 382577.8592253823, 'RMSE': 618.5287860927592, 'R2': 0.9547167607671987}
Linear_KBest {'model': 'Linear_KBest', 'selected_features': 'invoice_discount_amount,invoice_tax_amount,days_overdue', 'MAE': 780.1886112606463, 'MSE': 1067987.843629956, 'RMSE': 1033.4349731018183, 'R2': 0.8735892633234476}
Tree_Forward {'model': 'Tree_Forward', 'selected_features': 'collection_type,invoice_tax_amount,days_overdue', 'MAE': 296.36696187947285, 'MSE': 181334.21800918283, 'RMSE': 425.83355669695976, 'R2': 0.9785366544947774}
Tree_KBest {'model': 'Tree_KBest', 'selected_features': 'invoice_discount_amount,invoice_tax_amount,days_overdue', 'MAE': 430.88720970587093, 'MSE': 406865.2846554687, 'RMSE': 637.8599255757244, 'R2': 0.9518420170527394}


,model,selected_features,MAE,MSE,RMSE,R2
2,Tree_Forward,"collection_type,invoice_tax_amount,days_overdue",296.366962,1.813342e+05,425.833557,0.978537
0,Linear_Forward,"collection_type,invoice_tax_amount,days_overdue",488.746154,3.825779e+05,618.528786,0.954717
3,Tree_KBest,"invoice_discount_amount,invoice_tax_amount,day...",430.887210,4.068653e+05,637.859926,0.951842
1,Linear_KBest,"invoice_discount_amount,invoice_tax_amount,day...",780.188611,1.067988e+06,1033.434973,0.873589


## 4. One final test evaluation and a mean baseline

In [5]:
best_name = comparison.iloc[0]['model']
final_encoder = OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
X_dev = encode_frame(df.iloc[dev_idx], final_encoder, fit=True)
X_test = encode_frame(df.iloc[test_idx], final_encoder)
y_dev = df.expected_payment_amount.iloc[dev_idx].to_numpy()
y_test = df.expected_payment_amount.iloc[test_idx].to_numpy()
final_selector = clone(candidates[best_name][0]).fit(X_dev,y_dev)
final_model = clone(candidates[best_name][1]).fit(final_selector.transform(X_dev), y_dev)
test_prediction = final_model.predict(final_selector.transform(X_test))
test_rows=[{'model':best_name,'split':'random_test',**metrics(y_test,test_prediction)},
           {'model':'Training_mean','split':'random_test',**metrics(y_test,np.full(len(y_test),y_dev.mean()))}]
print(pd.DataFrame(test_rows).to_string(index=False))
pd.DataFrame(test_rows).to_csv(RESULTS / 'regression_test_metrics.csv',index=False)
with open(ARTIFACTS / 'selected_regressor.pkl','wb') as f:
    pickle.dump({'encoder':final_encoder,'selector':final_selector,'model':final_model,'features':features,'name':best_name},f)
pd.DataFrame({'row_index':test_idx,'actual':y_test,'prediction':test_prediction}).to_csv(RESULTS/'regression_test_predictions.csv.gz',index=False,compression={'method':'gzip','mtime':0})
split_info={'seed':SEED,'rows':len(df),'train':len(train_idx),'validation':len(val_idx),'test':len(test_idx),'selected_model':best_name}
(RESULTS/'regression_split.json').write_text(json.dumps(split_info,indent=2))

        model       split         MAE          MSE        RMSE        R2
 Tree_Forward random_test  295.043522 1.721469e+05  414.905886  0.979442
Training_mean random_test 1068.136717 8.373553e+06 2893.709167 -0.000002


131

## 5. Exploratory RNN and LSTM experiments
All rows are sorted by issue time. Target-row boundaries define a 70% fitting, 10% validation, 20% test split. Ten preceding feature rows predict the next invoice payment. Encoders and feature/target scalers fit only the fitting partition. Validation and test windows may include earlier observed rows, but never future features or target values. Windows overlapping at split boundaries reflect available history rather than independent observations.

The original positional indexing bug is corrected by converting targets to NumPy arrays. The original arbitrary deletion of 100,000 rows is removed. Training uses at most 30 epochs, batch size 512 and validation early stopping with patience 3. The best validation weights are restored. A fitting-partition mean provides a baseline. Weak results are retained rather than presented as successful forecasting.

In [6]:
import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)
ordered = df.sort_values('issue_date',kind='stable').reset_index(drop=True)
fit_end = int(len(ordered)*.7); val_end=int(len(ordered)*.8)
sequence_encoder=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
fit_frame=encode_frame(ordered.iloc[:fit_end], sequence_encoder, fit=True)
full_frame=encode_frame(ordered, sequence_encoder)
feature_scaler=MinMaxScaler().fit(fit_frame)
target_scaler=StandardScaler().fit(ordered[['expected_payment_amount']].iloc[:fit_end])
features_scaled=feature_scaler.transform(full_frame).astype('float32')
targets_scaled=target_scaler.transform(ordered[['expected_payment_amount']]).ravel().astype('float32')
timesteps=10
windows=np.lib.stride_tricks.sliding_window_view(features_scaled,timesteps,axis=0)[:-1].transpose(0,2,1).copy()
target_positions=np.arange(timesteps,len(ordered))
sequence_targets=targets_scaled[timesteps:]
masks=[target_positions<fit_end,(target_positions>=fit_end)&(target_positions<val_end),target_positions>=val_end]
seq_X=[windows[mask] for mask in masks]; seq_y=[sequence_targets[mask] for mask in masks]
del windows
print('Sequence train/validation/test:', [len(x) for x in seq_y], 'Input:',seq_X[0].shape, flush=True)
options=tf.data.Options(); options.threading.private_threadpool_size=1; options.threading.max_intra_op_parallelism=1
def dataset(x,y):
    return tf.data.Dataset.from_tensor_slices((x,y)).batch(512).with_options(options)
train_ds=dataset(seq_X[0],seq_y[0]); val_ds=dataset(seq_X[1],seq_y[1])
test_features=tf.data.Dataset.from_tensor_slices(seq_X[2]).batch(512).with_options(options)
true_sequence=target_scaler.inverse_transform(seq_y[2].reshape(-1,1)).ravel()
neural_rows=[]

Sequence train/validation/test: [372429, 53206, 106412] Input: (372429, 10, 6)


In [7]:
for name, layer in [('RNN',tf.keras.layers.SimpleRNN(50,activation='tanh')),('LSTM',tf.keras.layers.LSTM(64,activation='tanh'))]:
    tf.keras.utils.set_random_seed(SEED)
    model=tf.keras.Sequential([tf.keras.Input(shape=seq_X[0].shape[1:]),layer])
    if name=='LSTM': model.add(tf.keras.layers.Dropout(.2))
    model.add(tf.keras.layers.Dense(1))
    model.compile(optimizer='adam',loss='mse')
    callback=tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)
    history=model.fit(train_ds,validation_data=val_ds,epochs=30,callbacks=[callback],verbose=2)
    pred_scaled=model.predict(test_features,verbose=0).ravel()
    pred=target_scaler.inverse_transform(pred_scaled.reshape(-1,1)).ravel()
    row={'model':name,'split':'chronological_test','epochs':len(history.history['loss']),**metrics(true_sequence,pred)}
    neural_rows.append(row); print(row,flush=True)
    model.save(ARTIFACTS/(name.lower()+'.keras'))
    pd.DataFrame(history.history).to_csv(RESULTS/(name.lower()+'_training_history.csv'),index=False)
baseline=np.full(len(true_sequence),ordered.expected_payment_amount.iloc[:fit_end].mean())
neural_rows.append({'model':'Fitting_mean','split':'chronological_test','epochs':0,**metrics(true_sequence,baseline)})
pd.DataFrame(neural_rows).to_csv(RESULTS/'sequence_test_metrics.csv',index=False)
with open(ARTIFACTS/'sequence_preprocessing.pkl','wb') as f:
    pickle.dump({'encoder':sequence_encoder,'feature_scaler':feature_scaler,'target_scaler':target_scaler,'features':features,'timesteps':timesteps},f)
sequence_info={'rows':len(ordered),'fitting_row_end':fit_end,'validation_row_end':val_end,'timesteps':timesteps,'sequence_counts':[len(y) for y in seq_y]}
(RESULTS/'sequence_split.json').write_text(json.dumps(sequence_info,indent=2))
pd.DataFrame(neural_rows)

Epoch 1/30
728/728 - 4s - 5ms/step - loss: 1.0061 - val_loss: 0.9867
Epoch 2/30
728/728 - 3s - 4ms/step - loss: 1.0023 - val_loss: 0.9856
Epoch 3/30
728/728 - 3s - 4ms/step - loss: 1.0018 - val_loss: 0.9854
Epoch 4/30
728/728 - 3s - 4ms/step - loss: 1.0015 - val_loss: 0.9853
Epoch 5/30
728/728 - 2s - 3ms/step - loss: 1.0013 - val_loss: 0.9852
Epoch 6/30
728/728 - 2s - 3ms/step - loss: 1.0011 - val_loss: 0.9852
Epoch 7/30
728/728 - 2s - 3ms/step - loss: 1.0010 - val_loss: 0.9851
Epoch 8/30
728/728 - 2s - 3ms/step - loss: 1.0009 - val_loss: 0.9851
Epoch 9/30
728/728 - 2s - 3ms/step - loss: 1.0008 - val_loss: 0.9851
Epoch 10/30
728/728 - 3s - 4ms/step - loss: 1.0007 - val_loss: 0.9852
Epoch 11/30
728/728 - 3s - 4ms/step - loss: 1.0007 - val_loss: 0.9852
{'model': 'RNN', 'split': 'chronological_test', 'epochs': 11, 'MAE': 1076.1651611328125, 'MSE': 9598030.0, 'RMSE': 3098.0687532719476, 'R2': -0.0007604360580444336}
Epoch 1/30
728/728 - 10s - 14ms/step - loss: 1.0012 - val_loss: 0.9849
Epo

/home/agent/.local/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/home/agent/.local/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


,model,split,epochs,MAE,MSE,RMSE,R2
0,RNN,chronological_test,11,1076.165161,9.598030e+06,3098.068753,-0.000760
1,LSTM,chronological_test,15,1083.472412,9.592386e+06,3097.157729,-0.000172
2,Fitting_mean,chronological_test,0,1084.316984,9.592263e+06,3097.137949,-0.000159
